In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080210

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] exp01
[学号] 20244080210
[机器指纹] 90cf8d31534ddeec
[RUN_ID] 20244080210-90cf8d31534ddeec
[Python] 3.11.17


In [2]:
import re
from bs4 import BeautifulSoup

# 实验1：正则表达式的基础匹配

## 1.1 匹配数字、空白和独立单词 cat

\d+ 匹配一个或多个数字；\s+ 匹配一个或多个连续空白字符；\bcat\b 中 \b 是单词边界，只匹配独立单词 cat，不会匹配 cats。

In [3]:
text1 = "cat on mat; later: cats and dogs"

print("数字 \\d+ :", re.findall(r"\d+", text1))
print("空白 \\s+ :", re.findall(r"\s+", text1))
print("独立 cat \\bcat\\b :", re.findall(r"\bcat\b", text1))

数字 \d+ : []
空白 \s+ : [' ', ' ', ' ', ' ', ' ', ' ']
独立 cat \bcat\b : ['cat']


## 1.2 交替匹配与顺序差异

cat|dog 表示匹配 cat 或 dog。正则默认区分大小写，所以 CAT 不会被匹配。a|aa|aaa 和 aaa|aa|a 的差异说明：交替分支从左到右尝试，前面的分支优先匹配。

In [4]:
text2 = "cat dog bird CAT"
print("cat|dog :", re.findall(r"cat|dog", text2))

print('a|aa|aaa 在 "aaaa" 上 :', re.findall(r"a|aa|aaa", "aaaa"))
print('aaa|aa|a 在 "aaaa" 上 :', re.findall(r"aaa|aa|a", "aaaa"))

cat|dog : ['cat', 'dog']
a|aa|aaa 在 "aaaa" 上 : ['a', 'a', 'a', 'a']
aaa|aa|a 在 "aaaa" 上 : ['aaa', 'a']


a|aa|aaa 先尝试 a，所以每次只匹配一个 a；aaa|aa|a 先尝试最长的 aaa，所以先匹配 aaa，剩下一个 a。

## 1.3 量词匹配

* 表示前面的字符出现 0 次或多次；+ 表示至少 1 次；? 表示 0 次或 1 次；{2} 表示恰好 2 次；{1,2} 表示 1 到 2 次。

In [5]:
text3 = "ct cat caat caaat"

patterns = [
    (r"ca*t", "c + a出现0次或多次 + t"),
    (r"ca+t", "c + a出现至少1次 + t"),
    (r"ca?t", "c + a出现0次或1次 + t"),
    (r"ca{2}t", "c + a恰好2次 + t"),
    (r"ca{1,2}t", "c + a出现1到2次 + t"),
]

for p, desc in patterns:
    print(f"{p:10s} -> {re.findall(p, text3)}    # {desc}")

ca*t       -> ['ct', 'cat', 'caat', 'caaat']    # c + a出现0次或多次 + t
ca+t       -> ['cat', 'caat', 'caaat']    # c + a出现至少1次 + t
ca?t       -> ['ct', 'cat']    # c + a出现0次或1次 + t
ca{2}t     -> ['caat']    # c + a恰好2次 + t
ca{1,2}t   -> ['cat', 'caat']    # c + a出现1到2次 + t


# 实验2：分组、回溯引用与替换

## 2.1 提取邮箱用户名和域名

用两个捕获组：第 1 组匹配用户名，第 2 组匹配域名。finditer() 可以逐个输出完整邮箱、用户名和域名。

In [6]:
text_email = "联系: alice@example.com; 备选: bob@test.org"
email_pat = r"([A-Za-z0-9._%+-]+)@([A-Za-z0-9.-]+\.[A-Za-z]{2,})"

print("findall 结果:", re.findall(email_pat, text_email))
for m in re.finditer(email_pat, text_email):
    print("完整邮箱:", m.group(0))
    print("用户名:", m.group(1))
    print("域名:", m.group(2))
    print("-" * 30)

findall 结果: [('alice', 'example.com'), ('bob', 'test.org')]
完整邮箱: alice@example.com
用户名: alice
域名: example.com
------------------------------
完整邮箱: bob@test.org
用户名: bob
域名: test.org
------------------------------


## 2.2 后向引用找连续重复单词

\b(\w+)\s+\1\b 中 (\w+) 捕获一个单词，\s+ 匹配中间空白，\1 后向引用第 1 组，要求后面出现同一个单词。\b 保证单词边界。

In [7]:
text_dup = "cat cat, dog cat, bird bird"
dup_pat = r"\b(\w+)\s+\1\b"

print("重复单词:", re.findall(dup_pat, text_dup))

for m in re.finditer(dup_pat, text_dup):
    print("完整匹配:", m.group(0), "| 重复词:", m.group(1))

重复单词: ['cat', 'bird']
完整匹配: cat cat | 重复词: cat
完整匹配: bird bird | 重复词: bird


## 2.3 日期格式转换：年-月-日 → 月-日-年

正则 (\d{4})([-/.])(\d{2})\2(\d{2}) 中：第 1 组是年，第 2 组是分隔符，第 3 组是月，\2 要求第二个分隔符与第一个相同，第 4 组是日。替换表达式 \3\2\4\2\1 表示“月 + 原分隔符 + 日 + 原分隔符 + 年”。

In [8]:
text_dates = "2024-05-01\n2024/05/02\n2024.05.03"
date_pat = r"(\d{4})([-/.])(\d{2})\2(\d{2})"

new_dates = re.sub(date_pat, r"\3\2\4\2\1", text_dates)

print("转换前：")
print(text_dates)
print("\n转换后：")
print(new_dates)
print("\n转换前行数:", len(text_dates.splitlines()))
print("转换后行数:", len(new_dates.splitlines()))

转换前：
2024-05-01
2024/05/02
2024.05.03

转换后：
05-01-2024
05/02/2024
05.03.2024

转换前行数: 3
转换后行数: 3


## 2.4 电话号码统一格式

用三个捕获组分别匹配 3 位区号、3 位中间号、4 位尾号。替换表达式 (\1) \2-\3 把结果统一成 (123) 456-7890 形式。

In [9]:
text_phone = "联系我: 123-456-7890 或 987.654.3210"
phone_pat = r"(\d{3})[-.](\d{3})[-.](\d{4})"

new_phone = re.sub(phone_pat, r"(\1) \2-\3", text_phone)
print(new_phone)

联系我: (123) 456-7890 或 (987) 654-3210


# 实验3：使用 BeautifulSoup 解析 HTML

## 3.1 构建解析树，输出标题和 h1

BeautifulSoup(html_doc, "html.parser") 用 Python 标准库解析器构建解析树。soup.title.get_text() 获取标题文本，soup.h1.name 获取 h1 标签名，soup.h1.get_text() 获取 h1 文本。

In [10]:
html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲: 正则表达式</p>
<p class="course">第七讲: 结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

print("页面标题文本:", soup.title.get_text())
print("h1 标签名:", soup.h1.name)
print("h1 文本:", soup.h1.get_text())

页面标题文本: 实验课程
h1 标签名: h1
h1 文本: 数据分析


## 3.2 find_all 查找课程段落和链接

find_all("p", class_="course") 找所有 class 为 course 的段落。因为 class 是 Python 关键字，所以 BeautifulSoup 中使用 class_。链接文字用 get_text()，href 通过属性 a["href"] 获取。

In [11]:
courses = soup.find_all("p", class_="course")
print("class='course' 的段落数量:", len(courses))
for i, p in enumerate(courses, 1):
    print(f"段落{i}: {p.get_text()}")

links = soup.find_all("a")
print("\n链接数量:", len(links))
for i, a in enumerate(links, 1):
    print(f"链接{i} 文字: {a.get_text()} | href: {a['href']}")

class='course' 的段落数量: 2
段落1: 第六讲: 正则表达式
段落2: 第七讲: 结构化数据

链接数量: 2
链接1 文字: 讲义六 | href: https://example.org/lecture06
链接2 文字: 讲义七 | href: https://example.org/lecture07


## 3.3 get_text() 提取可见文本并比较

get_text() 会去掉 HTML 标签，只保留文本。separator="\n" 控制不同文本片段之间用换行连接，strip=True 去掉首尾空白。整个文档会包含标题、h1、段落和链接文字；单个段落只包含该段落文字；单个链接只包含链接锚文本，不包含 href 属性。

In [12]:
whole_text = soup.get_text(separator="\n", strip=True)
p_text = courses[0].get_text(strip=True)
a_text = links[0].get_text(strip=True)

print("整个文档可见文本：")
print(whole_text)

print("\n单个段落文本:", repr(p_text))
print("单个链接文本:", repr(a_text))

print("\n比较：")
print("整个文档包含所有可见文本；单个段落只包含该段落；单个链接只包含链接文字，不包含 href。")

整个文档可见文本：
实验课程
数据分析
第六讲: 正则表达式
第七讲: 结构化数据
讲义六
讲义七

单个段落文本: '第六讲: 正则表达式'
单个链接文本: '讲义六'

比较：
整个文档包含所有可见文本；单个段落只包含该段落；单个链接只包含链接文字，不包含 href。
